In [ ]:
%load_ext autoreload
%autoreload 2
%matplotlib inline
import os
from pathlib import Path
from dotenv import load_dotenv
load_dotenv()
import numpy as np
from tidy3d import web

import helpers as H   # cylinder geometry, simulation and submission (this folder)

In [ ]:
tidy3dAPI = os.environ["API_TIDY3D_KEY"]
web.configure(tidy3dAPI)

In [ ]:
lambdas = np.array([7,2])

In [ ]:
# The LSU network is built from tidy3d cylinders (one per rod) instead of the 1000^3 voxel h5,
# which the tidy3d server cannot load. Same rods, radius and periodic box as the voxelizer
# (20250903_create_h5_from_ends.ipynb -> N100000/n_2.90_ff_0.2248_periodic.h5).
rods_file = Path("../../Structures/53.10x 53.10x 53.10_n100000_healed_rodlen_ends_unclipped.txt")
box_size = 53.10        # periodic box edge (um)
radius = 0.335          # rod radius (um)
n = 2.9
ff = 0.2248             # label only: ff of the voxelized periodic h5 built from these rods
n_eff = 1               # background index
project_name = "20261124_LSU_T_L_Tests"
runtime_ps = 70e-12
min_steps_per_lambda = 20
# cuts=[0.02,0.05,0.1,0.15,0.2,0.25,0.3,0.35,0.4,0.45,0.5,0.55,0.6,0.65,0.7,0.75,0.8,0.85,0.9,0.95,1]
cuts = [1]
run = False
# Task-id files are stored in the repo-level data folder: /mnt/fast/Codes/tidy3d/data/<project_name>/...
# Run files hold "\n<task id>" (empty ref line, read with get_ref=False); the reference is in LSU_reference.txt.
project_dir = Path("/mnt/fast/Codes/tidy3d/data") / project_name
folder_desc = project_dir / f"n_{n:.2f}"
folder_desc.mkdir(parents=True, exist_ok=True)

# Single empty-domain reference for the whole project. It is valid for every cut because
# Lz is fixed. Use a new project_name if lambdas, runtime or box size change.
ref_file = project_dir / "LSU_reference.txt"   # "\n<ref id>" -> load with get_ref=False

In [ ]:
# Rods plus their periodic images across the box faces (periodic in x, y and z, like the h5).
rods = H.add_periodic_images(H.load_rods(rods_file), box_size, radius)
print(f"{len(rods)} rods incl. periodic images")

In [ ]:
for cut in cuts:
    sim_name = f"LSU_cyl_n_{n:.2f}_ff_{ff:.4f}_size_{cut}"
    run_file = folder_desc / f"{sim_name}.txt"   # "\n<task id>" (no own reference)
    if run_file.exists():
        print(f"Exist! {sim_name}")
        continue
    try:
        slab_rods, t_slab = H.cut_slab(rods, box_size, cut, radius)
        sim = H.build_simulation(slab_rods, t_slab, radius=radius, box=box_size, lambda_range=lambdas,
                                 runtime_ps=runtime_ps, min_steps_per_lambda=min_steps_per_lambda,
                                 permittivity=n**2, n_freqs=400, shutoff=1e-9, absorbers=130,
                                 bg_permittivity=n_eff**2)
        print(f"{sim_name}: {len(slab_rods)} cylinders, slab {t_slab:.2f} um, "
              f"{sim.num_cells/1e6:.0f} M cells, {sim.num_time_steps} time steps")
        if not run:
            H.plot_layout(sim)
        H.submit(sim, project_name=project_name, sim_name=sim_name,
                 run_file=run_file, ref_file=ref_file, run=run)
    except Exception as e:
        print(f"Error processing {sim_name}: {e}")